In [86]:
!pip install timm  monai 

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [55]:
import monai
print(monai.__version__)


1.4.0


In [57]:
from monai.transforms import (
    Activationsd,
    Compose,
    LoadImage,
    EnsureChannelFirstD,
    #AsChannelFirstd,
    AsChannelLastd,
    AsDiscreted,
    EnsureTyped,
    LoadImaged,
    NormalizeIntensityd,
    Resized,
    Spacingd,
    ToNumpyd,
    SaveImaged,
    EnsureChannelFirstd,
    ScaleIntensityd,
    ResizeWithPadOrCropd,
    KeepLargestConnectedComponentd,
    Invertd,
)
from monailabel.transform.post import (Restored, DumpImagePrediction2Dd,)

In [60]:
import torch
import monai
from lib.models import PolypPVT
from monai.inferers import SimpleInferer
from lib.infers import Usg
from lib.transforms import (SaveImagePred, PVTNetSumOutd,)
from monailabel.tasks.infer.basic_infer import BasicInferTask
from typing import Dict
print(torch.__version__)  # Wersja PyTorch
print(monai.__version__)  # Wersja MONAI


2.6.0+cu124
1.4.0


In [ ]:
#from PIL import Image
#img = Image.open("../images_rgb/cg010321-105954__002.png")
#img.show()


In [79]:
# Define transforms for image preprocessing
transforms = Compose([
    LoadImaged(keys="image"),
    EnsureChannelFirstd(keys="image"),
    ScaleIntensityd(keys="image"),
    Resized(keys="image",spatial_size=(512,512),mode="nearest"	) ,
    #SaveImaged(keys="image", output_dir="./output", output_ext=".png", output_postfix="processed", resample=False)
])
data = {"image": "../images_rgb/cg010321-105954__002.png"} 
# Apply transforms to image
image = transforms(data) #mages_rgb/cg010321-105954__002.png")

In [80]:
conf = {"param1": "value1", "param2": "value2"} 
# Tworzenie obiektu klasy Usg
usg_task = Usg("./lib/models/pvt_v2_b2.pth",PolypPVT,conf)


{'param1': 'value1', 'param2': 'value2'}


In [103]:
 # Wywołanie metody inferera
inferer = usg_task.inferer()
image_tensor = image['image'].unsqueeze(0)
print(image_tensor)
network=PolypPVT(channel=32)
#network.eval()
network.load_state_dict(torch.load('./model/pretrained_usg.pt'),strict=False)
output = inferer(image_tensor,network)

metatensor([[[[0.5644, 0.7376, 0.8614,  ..., 0.0000, 0.0000, 0.0000],
          [0.5644, 0.7327, 0.8564,  ..., 0.0000, 0.0000, 0.0000],
          [0.5594, 0.7079, 0.8416,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.5347, 0.7228, 0.8960,  ..., 0.0000, 0.0000, 0.0000],
          [0.5545, 0.7327, 0.8960,  ..., 0.0000, 0.0000, 0.0000],
          [0.5842, 0.7525, 0.9109,  ..., 0.0000, 0.0000, 0.0000]],

         [[0.5644, 0.7376, 0.8614,  ..., 0.0000, 0.0000, 0.0000],
          [0.5644, 0.7327, 0.8564,  ..., 0.0000, 0.0000, 0.0000],
          [0.5594, 0.7079, 0.8416,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.5347, 0.7228, 0.8960,  ..., 0.0000, 0.0000, 0.0000],
          [0.5545, 0.7327, 0.8960,  ..., 0.0000, 0.0000, 0.0000],
          [0.5842, 0.7525, 0.9109,  ..., 0.0000, 0.0000, 0.0000]],

         [[0.5644, 0.7376, 0.8614,  ..., 0.0000, 0.0000, 0.0000],
          [0.5644, 0.7327, 0.8564,  ..., 0.0000, 0.0000, 0.0000],
          [0.5594, 0.7079, 0.8416,  ..

`nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.


In [104]:
print(output)


metatensor([[[[ -4.1544,  -4.1544,  -4.1544,  ...,  -4.4005,  -4.4005,  -4.4005],
          [ -4.1544,  -4.1544,  -4.1544,  ...,  -4.4005,  -4.4005,  -4.4005],
          [ -4.1544,  -4.1544,  -4.1544,  ...,  -4.4005,  -4.4005,  -4.4005],
          ...,
          [-12.0817, -12.0817, -12.0817,  ...,  -4.2822,  -4.2822,  -4.2822],
          [-12.0817, -12.0817, -12.0817,  ...,  -4.2822,  -4.2822,  -4.2822],
          [-12.0817, -12.0817, -12.0817,  ...,  -4.2822,  -4.2822,  -4.2822]]]],
       grad_fn=<AliasBackward0>)


In [105]:
 post_transforms = Compose([
   # LoadImaged(keys="image"),
    EnsureTyped(keys="pred", device=data.get("device") if data else None),
    Activationsd(keys="pred", sigmoid=True),
    AsDiscreted(keys="pred",  threshold=0.5),
    KeepLargestConnectedComponentd(keys="pred"),
    #SaveImagePred(keys="pred"),
    #Restored(keys="pred", ref_image="image"),
    SaveImaged(keys="pred", output_dir="./output2", output_ext=".png", output_postfix="processed", resample=False),
    ##Restored(keys="pred", ref_image="image"),
    #AsChannelLastd(keys="pred"),
        ])
post_transforms_result = post_transforms({"pred": output})

pred = post_transforms_result["pred"]

2025-03-21 18:02:13,246 INFO image_writer.py:197 - writing: output2/cg010321-105954__002/cg010321-105954__002_processed.png


In [106]:
print(post_transforms_result)

{'pred': metatensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]])}
